In [1]:
from backend.Agents.PDF_create import 

SyntaxError: invalid syntax (3619130629.py, line 1)

In [7]:
import nltk

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\soumi\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\soumi\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\soumi\AppData\Roaming\nltk_data...


True

In [15]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from selenium import webdriver
from rank_bm25 import BM25Okapi
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
import requests
import trafilatura
from dotenv import load_dotenv
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()
client = OpenAI()

def get_words_list(text: str):
    stop_words = set(ENGLISH_STOP_WORDS)
    lemmatizer = WordNetLemmatizer()

    # Fix merged words (camelCase → split)
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)

    # Separate numbers from words
    text = re.sub(r'(\d+)', r' \1 ', text)

    # Lowercase
    text = text.lower()

    # Extract words
    words = re.findall(r'\b[a-zA-Z]+\b', text)

    # Remove stopwords + lemmatize
    final_words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words and len(word) > 2
    ]

    return final_words

def keyword_score(sentence : str, body:str):
    sentence_list=get_words_list(sentence)

    body_word_list=get_words_list(body)
    sentence_set=set(sentence_list)
    score=0
    for a in body_word_list:
        if any(a.startswith(b) or b.startswith(a) for b in sentence_set):
            #checking prefix between a & b
            score+=1  

    final_score=score/len(body_word_list)

    return final_score 

def filter_list(query, A_dict_list):
    filtered_list=[]

    for a_dict in A_dict_list:

        body=a_dict.get('body')
        title=a_dict.get('title')

        score1=keyword_score(query, body)
        score2=keyword_score(query, title)

        score=0.95*score1+0.05*score2
        a_dict['keyword_score']=score
        
        if score>0.10:
            filtered_list.append(a_dict)

    filtered_list=sorted(filtered_list, key=lambda x: x['keyword_score'], reverse=True)
    return filtered_list


def get_html_content(url):
    headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
    html=requests.get(url, headers=headers).text
    content=trafilatura.extract(html)

    if not content or len(content) < 150:
        driver = webdriver.Chrome()
        driver.get(url)
        html = driver.page_source
        content=trafilatura.extract(html)
        driver.quit()
    return str(content)

def chunk_text(text):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,       
        chunk_overlap=50,      
        separators=["\n\n", "\n", ".", " "]
    )
    
    chunks = splitter.split_text(text)
    return chunks


def bm25_retrieve(text:str, query:str):
    chunks = chunk_text(text)
    
    tokenized_corpus = [get_words_list(s) for s in chunks]
    
    bm25 = BM25Okapi(tokenized_corpus)
    
    tokenized_query = get_words_list(query)
    
    scores = bm25.get_scores(tokenized_query)
    
    # Rank sentences
    ranked = sorted(
        zip(chunks, scores),
        key=lambda x: x[1],
        reverse=True
    )
    ranked=ranked[:40]
    ranked_sentences=[sentence[0] for sentence in ranked]
    return ranked_sentences


def get_webpage(query:str, A_dict_list: list):

    filtered_list=filter_list(query=query, A_dict_list=A_dict_list)

    if len(filtered_list)>4:
        filtered_list=filtered_list[:4]

    web_content_chunks={}
    
    for filter in filtered_list:
        semantic_similar_sentence=[]
        website=str(filter.get('href'))
        
        content=get_html_content(website)

        sentences=bm25_retrieve(text=content, query=query)

        response = client.embeddings.create(
                model="text-embedding-3-small",
                input=[query] + sentences
                )
        
        embeddings = [item.embedding for item in response.data]
        query_embedding = embeddings[0]
        sentence_embeddings = embeddings[1:]
        scores = cosine_similarity([query_embedding], sentence_embeddings)[0]

        results = sorted(
            zip(sentences, scores),
            key=lambda x: x[1],
            reverse=True)
        
        web_content_chunks[website]=results[:7]

    
    return web_content_chunks
    


In [7]:
from backend.Agents.Search_Agent import search
query='What is the market rate of Gold Today in India'

A_dict_list=search(query)

In [10]:
a=get_webpage(query=query, A_dict_list=A_dict_list)

In [14]:
a

{'https://www.kalyanjewellers.net/kalyan_gold_rates/gold-rate/todays-gold-rate-in-karol-bagh': [('Discover the 22K gold rate in Delhi-Karol Bagh. Presenting you the latest updates on gold prices. Check out our gold price listings to find more.\n*Price may vary by city',
   np.float64(0.6034856971937813))],
 'https://www.prokerala.com/finance/gold-price.php?currency=INR': [("Live Gold Price in India • Indian Rupee\nToday's Gold Price in India = 14167.16996 INR / 1 Gram*\nLive Gold price in India on March 31 2026 stands at ₹13754.53 per gram for 22K gold and ₹14909.91 per gram for 24K gold in INR.",
   np.float64(0.7352927017499935)),
  ('The gold price today in India is ₹ 14167.17 per gram.\nThe average price for March 2026 is\n₹ 14399.14,\nshowing a lower trend compared to the March 2026 average of ₹ 14526.06.\nThe highest price for March 2026 reached ₹ 15569.88 , while the lowest was ₹12791.41 .',
   np.float64(0.7340484386581858)),
  ('Additionally, use the Currency Converter to esti